# 03 — Exploratory Data Analysis: Cyber Threat Patterns & Insights

**Cybersecurity Threat Intelligence Analytics Project**  
**Dataset:** CICIDS-2017 — 2,574,264 network flow records across 5 capture days  

---

## Objective

This notebook performs deep exploratory data analysis on the cleaned dataset to uncover:
- **Attack distribution patterns** — Which attack types and families dominate?
- **Behavioral differences** — How does malicious traffic differ from benign?
- **Port targeting patterns** — Which services are most targeted?
- **Temporal patterns** — How did the simulated attack campaign unfold over 5 days?
- **Feature correlations** — Which features are redundant? Which are discriminative?
- **Outlier characteristics** — Where do extreme values cluster?

These insights will directly inform feature engineering and model selection in later notebooks.

> **Interview Relevance (Deloitte):** EDA demonstrates analytical thinking, domain understanding, 
> and the ability to translate data patterns into actionable cybersecurity recommendations.

## 1. Setup & Configuration

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

# ── Style ────────────────────────────────────────────────────────────
plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor': '#1a1a2e',
    'axes.facecolor':   '#16213e',
    'axes.edgecolor':   '#e94560',
    'axes.labelcolor':  '#ffffff',
    'text.color':       '#ffffff',
    'xtick.color':      '#cccccc',
    'ytick.color':      '#cccccc',
    'grid.color':       '#2a2a4a',
    'grid.alpha':       0.4,
    'font.size':        11,
    'axes.titlesize':   14,
    'axes.labelsize':   12,
    'figure.dpi':       100,
})

# ── Paths ────────────────────────────────────────────────────────────
DATA_PATH = Path('../data/processed/flows_clean.parquet')
FIG_DIR   = Path('../reports/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

SAVE_KW = dict(dpi=150, bbox_inches='tight', facecolor='#1a1a2e')

# ── Color Maps ───────────────────────────────────────────────────────
FAMILY_COLORS = {
    'DoS':          '#e94560',
    'DDoS':         '#ff6f61',
    'PortScan':     '#0f3460',
    'Brute Force':  '#00d2ff',
    'Web Attack':   '#ffc300',
    'Bot':          '#7b2ff7',
    'Infiltration': '#00e676',
    'Heartbleed':   '#ff1744',
}

LABEL_COLORS = {
    'DoS Hulk':                  '#e94560',
    'DDoS':                      '#ff6f61',
    'PortScan':                  '#0f3460',
    'DoS GoldenEye':             '#ff8a80',
    'FTP-Patator':               '#00d2ff',
    'DoS Slowloris':             '#f48fb1',
    'DoS Slowhttptest':          '#ce93d8',
    'SSH-Patator':               '#4dd0e1',
    'Bot':                       '#7b2ff7',
    'Web Attack - Brute Force':  '#ffc300',
    'Web Attack - XSS':          '#ffd54f',
    'Infiltration':              '#00e676',
    'Web Attack - SQL Injection':'#ffab40',
    'Heartbleed':                '#ff1744',
}

print('✅ Setup complete — ready for EDA')

In [ ]:
# ── Load Data ────────────────────────────────────────────────────────
df = pd.read_parquet(DATA_PATH)
print(f'Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Benign flows:  {(~df["is_attack"]).sum():,}')
print(f'Attack flows:  {df["is_attack"].sum():,}')
print(f'Attack ratio:  {df["is_attack"].mean():.2%}')
print(f'Capture dates: {df["capture_date"].nunique()}')
print(f'Labels:        {df["observed_label"].nunique()}')
print(f'Families:      {df["attack_family"].nunique()}')

---
## 2. Attack Distribution Analysis

Understanding the **class distribution** is the first step in any classification problem.  
In cybersecurity, this tells us:
- Which attacks are most **prevalent** (highest risk)
- Which attacks are **rare** (hardest to detect, potentially most dangerous)
- The degree of **class imbalance** we must handle during modelling

In [ ]:
# ── 2a: Attack counts by observed_label (exclude BENIGN) ────────────
attacks = df[df['is_attack']].copy()
label_counts = attacks['observed_label'].value_counts().sort_values()

fig, ax = plt.subplots(figsize=(12, 7))
colors = [LABEL_COLORS.get(lbl, '#888888') for lbl in label_counts.index]
bars = ax.barh(label_counts.index, label_counts.values, color=colors, edgecolor='white', linewidth=0.5)

# Annotate counts
for bar, val in zip(bars, label_counts.values):
    ax.text(val + label_counts.max() * 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=10, color='white')

ax.set_xlabel('Number of Flows')
ax.set_title('Attack Counts by Observed Label (Excluding Benign)', fontsize=15, fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}K' if x >= 1000 else f'{x:.0f}'))
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
fig.savefig(FIG_DIR / '03_attack_counts_by_label.png', **SAVE_KW)
plt.show()
print(f'\nTotal attack flows: {len(attacks):,}')

In [ ]:
# ── 2b: Attack counts by attack_family ──────────────────────────────
family_counts = attacks['attack_family'].value_counts().sort_values()
family_pct = (family_counts / family_counts.sum() * 100).round(1)

fig, ax = plt.subplots(figsize=(10, 6))
colors = [FAMILY_COLORS.get(fam, '#888888') for fam in family_counts.index]
bars = ax.barh(family_counts.index, family_counts.values, color=colors, edgecolor='white', linewidth=0.5)

for bar, val, pct in zip(bars, family_counts.values, family_pct.values):
    ax.text(val + family_counts.max() * 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:,} ({pct}%)', va='center', fontsize=10, color='white')

ax.set_xlabel('Number of Flows')
ax.set_title('Attack Counts by Family', fontsize=15, fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}K' if x >= 1000 else f'{x:.0f}'))
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
fig.savefig(FIG_DIR / '03_attack_counts_by_family.png', **SAVE_KW)
plt.show()

In [ ]:
# ── 2c: Attacks per capture date by family (stacked bar) ────────────
date_family = attacks.groupby(['capture_date', 'attack_family']).size().unstack(fill_value=0)
# Reorder families by total count descending for legend
family_order = family_counts.sort_values(ascending=False).index.tolist()
date_family = date_family.reindex(columns=[f for f in family_order if f in date_family.columns], fill_value=0)

fig, ax = plt.subplots(figsize=(12, 7))
date_labels = [d.strftime('%Y-%m-%d\n(%A)') for d in date_family.index]
bottom = np.zeros(len(date_family))

for fam in date_family.columns:
    vals = date_family[fam].values
    ax.bar(date_labels, vals, bottom=bottom, label=fam,
           color=FAMILY_COLORS.get(fam, '#888'), edgecolor='white', linewidth=0.3)
    bottom += vals

ax.set_ylabel('Number of Attack Flows')
ax.set_title('Attack Volume by Family per Capture Date', fontsize=15, fontweight='bold')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', framealpha=0.7)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}K' if x >= 1000 else f'{x:.0f}'))
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
fig.savefig(FIG_DIR / '03_attacks_per_date_by_family.png', **SAVE_KW)
plt.show()

print('\n📊 Attack family breakdown per date:')
print(date_family.to_string())

### 🔍 Insight — Attack Distribution

| Family | Count | Share |
|--------|-------|-------|
| DoS | 193,748 | **45.5%** |
| DDoS | 128,016 | **30.0%** |
| PortScan | 90,819 | **21.3%** |
| Brute Force | 9,152 | 2.1% |
| Web Attack | 2,143 | 0.5% |
| Bot | 1,953 | 0.5% |
| Infiltration | 36 | <0.01% |
| Heartbleed | 11 | <0.01% |

**Key Takeaways:**
1. **DoS attacks dominate** (~45% of all attacks), followed by DDoS (~30%) and PortScan (~21%).
2. The top 3 families account for **96.8%** of all attacks — extreme class imbalance.
3. **Infiltration (36) and Heartbleed (11)** are ultra-rare — these will be very hard for supervised classifiers to learn. Anomaly detection may be more effective for these.
4. **Business Impact:** A SOC analyst would need to triage hundreds of thousands of volumetric (DoS/DDoS) alerts, making alert prioritization and aggregation critical.

---
## 3. Normal vs Malicious Traffic Comparison

A fundamental question for any IDS is: **How does attack traffic differ from normal traffic?**  
We compare key flow features between benign and malicious traffic to identify the most discriminative signals.

In [ ]:
# ── 3a: Side-by-side boxplots — Benign vs Attack ────────────────────
compare_features = [
    'flow_duration', 'total_packets', 'total_bytes',
    'bytes_per_second', 'fwd_packet_length_mean', 'bwd_packet_length_mean'
]

# Sample for performance (100K per class is enough for boxplots)
np.random.seed(42)
benign_sample = df[~df['is_attack']].sample(min(100_000, (~df['is_attack']).sum()), random_state=42)
attack_sample = df[df['is_attack']].sample(min(100_000, df['is_attack'].sum()), random_state=42)
sample_df = pd.concat([benign_sample, attack_sample])
sample_df['Traffic Type'] = sample_df['is_attack'].map({True: 'Attack', False: 'Benign'})
print(f'Using sample of {len(sample_df):,} flows (100K benign + 100K attack) for boxplots')

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
palette = {'Benign': '#00d2ff', 'Attack': '#e94560'}

for ax, feat in zip(axes.flat, compare_features):
    # Clip extreme values for visibility (show up to 99.5th percentile)
    upper = sample_df[feat].quantile(0.995)
    plot_data = sample_df[sample_df[feat] <= upper] if upper > 0 else sample_df
    
    sns.boxplot(data=plot_data, x='Traffic Type', y=feat, ax=ax,
                palette=palette, width=0.5, fliersize=1,
                boxprops=dict(alpha=0.8), medianprops=dict(color='yellow', linewidth=2))
    
    nice_name = feat.replace('_', ' ').title()
    ax.set_title(nice_name, fontsize=13, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('Benign vs Attack Traffic — Key Feature Distributions\n(clipped at 99.5th percentile for clarity)',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(FIG_DIR / '03_benign_vs_attack_boxplots.png', **SAVE_KW)
plt.show()

In [ ]:
# ── 3b: Log-scale distributions ─────────────────────────────────────
log_features = ['total_bytes', 'flow_duration', 'bytes_per_second', 'total_packets']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, feat in zip(axes.flat, log_features):
    for label, color, grp in [('Benign', '#00d2ff', benign_sample), ('Attack', '#e94560', attack_sample)]:
        vals = grp[feat].replace(0, np.nan).dropna()
        if len(vals) > 0:
            ax.hist(np.log1p(vals), bins=80, alpha=0.55, label=label, color=color, density=True)
    
    nice_name = feat.replace('_', ' ').title()
    ax.set_title(f'log₁ₚ({nice_name})', fontsize=13, fontweight='bold')
    ax.set_ylabel('Density')
    ax.legend(framealpha=0.7)
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('Log-Transformed Feature Distributions: Benign vs Attack',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(FIG_DIR / '03_log_distributions_benign_vs_attack.png', **SAVE_KW)
plt.show()

### 🔍 Insight — Benign vs Attack Traffic

**Key Discriminative Features:**
1. **Flow Duration:** Attack flows tend to be either very short (DoS Hulk, DDoS — rapid bursts) or very long (Slowloris, Slowhttptest — slow-rate attacks). Benign traffic has a more uniform distribution.
2. **Total Bytes:** Attack flows often show extreme values — either very high byte counts (volume-based attacks) or near-zero (reconnaissance/scanning).
3. **Bytes Per Second:** DoS/DDoS attacks push extremely high throughput. This feature alone provides strong discriminative power.
4. **Forward Packet Length Mean:** Benign traffic typically has larger forward packets (carrying application data), while many attacks send minimal payloads.
5. **Backward Packet Length Mean:** Response patterns differ — attacks often receive error responses or no response at all.

**Business Implication:** These features can be used to build real-time detection rules. For example, a flow with `bytes_per_second > threshold AND flow_duration < 1s` is a strong DoS indicator.

---
## 4. Port-wise Analysis

Understanding which **destination ports** (services) are targeted helps security teams:
- Prioritize **firewall rules** and **IDS signatures**
- Identify which services need **hardening**
- Detect **port scanning** reconnaissance activity

In [ ]:
# ── 4a: Top 20 Targeted Ports ───────────────────────────────────────
port_counts = attacks['destination_port'].value_counts().head(20).sort_values()

# Map well-known ports to service names
PORT_SERVICES = {
    21: 'FTP', 22: 'SSH', 25: 'SMTP', 53: 'DNS', 80: 'HTTP', 110: 'POP3',
    111: 'RPCbind', 135: 'MS-RPC', 139: 'NetBIOS', 143: 'IMAP',
    443: 'HTTPS', 444: 'SNPP', 445: 'SMB', 993: 'IMAPS', 995: 'POP3S',
    3306: 'MySQL', 5900: 'VNC', 8080: 'HTTP-Alt', 8888: 'HTTP-Alt2'
}

port_labels = [f'{p} ({PORT_SERVICES.get(p, "?")})' for p in port_counts.index]

fig, ax = plt.subplots(figsize=(12, 8))
colors = ['#e94560' if v > 10000 else '#00d2ff' if v > 1000 else '#888888' for v in port_counts.values]
bars = ax.barh(port_labels, port_counts.values, color=colors, edgecolor='white', linewidth=0.5)

for bar, val in zip(bars, port_counts.values):
    ax.text(val + port_counts.max() * 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=9, color='white')

ax.set_xlabel('Number of Attack Flows')
ax.set_title('Top 20 Targeted Destination Ports (Attack Traffic Only)',
             fontsize=15, fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}K' if x >= 1000 else f'{x:.0f}'))
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
fig.savefig(FIG_DIR / '03_top20_targeted_ports.png', **SAVE_KW)
plt.show()

In [ ]:
# ── 4b: Heatmap — Top 10 Ports × Attack Families ───────────────────
top10_ports = attacks['destination_port'].value_counts().head(10).index.tolist()
port_family_matrix = attacks[attacks['destination_port'].isin(top10_ports)].groupby(
    ['destination_port', 'attack_family']
).size().unstack(fill_value=0)

# Create nice row labels
port_family_matrix.index = [f'{p} ({PORT_SERVICES.get(p, "?")})' for p in port_family_matrix.index]

fig, ax = plt.subplots(figsize=(14, 7))
sns.heatmap(
    np.log1p(port_family_matrix),  # log scale for visibility
    annot=port_family_matrix.values,  # show raw counts
    fmt=',d',
    cmap='YlOrRd',
    linewidths=0.5,
    linecolor='#2a2a4a',
    ax=ax,
    cbar_kws={'label': 'log₁ₚ(Count)'},
    annot_kws={'fontsize': 9}
)
ax.set_title('Attack Family Distribution Across Top 10 Ports\n(cell values = raw counts, color = log scale)',
             fontsize=14, fontweight='bold')
ax.set_ylabel('Destination Port')
ax.set_xlabel('Attack Family')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
fig.savefig(FIG_DIR / '03_port_family_heatmap.png', **SAVE_KW)
plt.show()

### 🔍 Insight — Port Targeting Patterns

1. **Port 80 (HTTP)** is overwhelmingly the most targeted port with 324K+ attack flows — it receives DoS, DDoS, PortScan, Web Attacks, and Bot traffic. This makes sense as HTTP is the most exposed service.
2. **Port 21 (FTP)** and **Port 22 (SSH)** are targeted primarily by **Brute Force** attacks (FTP-Patator, SSH-Patator) — credential stuffing against remote access services.
3. **Port 8080 (HTTP-Alt)** receives significant attack traffic, often used for web application testing servers.
4. **The long tail** of ports (139, 445, 443, etc.) are targeted by **PortScan** reconnaissance — attackers probing for open services.

**Business Recommendation:** 
- Deploy a **Web Application Firewall (WAF)** on ports 80/8080
- Implement **rate limiting** and **fail2ban** on FTP/SSH services
- Use **port knocking** or **VPN** for sensitive management ports

---
## 5. Capture Date Trends

The CICIDS-2017 dataset was collected over **5 days** (July 3–7, 2017), with different attack types executed each day.  
Analyzing the temporal distribution reveals the **attack campaign schedule** and helps validate data integrity.

In [ ]:
# ── 5a: Grouped bar chart — Benign vs Attack families per date ──────
# Get benign counts per date
benign_by_date = df[~df['is_attack']].groupby('capture_date').size()
attack_by_date_family = attacks.groupby(['capture_date', 'attack_family']).size().unstack(fill_value=0)

# Combine into one DataFrame
combined = pd.DataFrame({'BENIGN': benign_by_date})
combined = combined.join(attack_by_date_family, how='outer').fillna(0).astype(int)

print('Daily breakdown:')
print(combined.to_string())
print()

In [ ]:
# ── 5b: Stacked bar showing composition per day ─────────────────────
fig, ax = plt.subplots(figsize=(14, 8))

# Plot attack families only (stacked)
date_labels = [d.strftime('%a\n%Y-%m-%d') for d in attack_by_date_family.index]
x = np.arange(len(date_labels))
bar_width = 0.35

# Benign bars
benign_vals = [benign_by_date.get(d, 0) for d in attack_by_date_family.index]
ax.bar(x - bar_width/2, benign_vals, bar_width, label='BENIGN',
       color='#00d2ff', edgecolor='white', linewidth=0.3, alpha=0.8)

# Attack stacked bars
bottom = np.zeros(len(x))
for fam in family_order:
    if fam in attack_by_date_family.columns:
        vals = attack_by_date_family[fam].values
        ax.bar(x + bar_width/2, vals, bar_width, bottom=bottom, label=fam,
               color=FAMILY_COLORS.get(fam, '#888'), edgecolor='white', linewidth=0.3)
        bottom += vals

ax.set_xticks(x)
ax.set_xticklabels(date_labels)
ax.set_ylabel('Number of Flows')
ax.set_title('Daily Traffic Composition: Benign vs Attack Families',
             fontsize=15, fontweight='bold')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', framealpha=0.7, fontsize=10)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}K' if x >= 1000 else f'{x:.0f}'))
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
fig.savefig(FIG_DIR / '03_daily_traffic_composition.png', **SAVE_KW)
plt.show()

In [ ]:
# ── 5c: Line chart — Total events & attack events per day ──────────
total_by_date = df.groupby('capture_date').size()
attack_count_by_date = attacks.groupby('capture_date').size()

fig, ax1 = plt.subplots(figsize=(12, 6))

date_x = [d.strftime('%a %m-%d') for d in total_by_date.index]

ax1.plot(date_x, total_by_date.values, 'o-', color='#00d2ff', linewidth=2.5,
         markersize=10, label='Total Flows', zorder=5)
ax1.plot(date_x, attack_count_by_date.reindex(total_by_date.index, fill_value=0).values,
         's-', color='#e94560', linewidth=2.5, markersize=10, label='Attack Flows', zorder=5)

# Fill between
ax1.fill_between(date_x, attack_count_by_date.reindex(total_by_date.index, fill_value=0).values,
                 alpha=0.15, color='#e94560')

# Annotate attack %
for i, d in enumerate(total_by_date.index):
    total = total_by_date[d]
    att = attack_count_by_date.get(d, 0)
    pct = att / total * 100
    ax1.annotate(f'{pct:.1f}% attack', (date_x[i], att),
                 textcoords='offset points', xytext=(0, 15),
                 fontsize=10, color='#e94560', fontweight='bold', ha='center')

ax1.set_ylabel('Number of Flows')
ax1.set_title('Daily Traffic Volume: Total vs Attack Flows', fontsize=15, fontweight='bold')
ax1.legend(fontsize=12, framealpha=0.7)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}K' if x >= 1000 else f'{x:.0f}'))
ax1.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(FIG_DIR / '03_daily_traffic_trend.png', **SAVE_KW)
plt.show()

### 🔍 Insight — Temporal Attack Campaign

The CICIDS-2017 experiment followed a deliberate **attack schedule**:

| Day | Date | Attack Types |
|-----|------|--------------|
| Monday | 2017-07-03 | **Benign only** (baseline day) |
| Tuesday | 2017-07-04 | **FTP-Patator, SSH-Patator** (Brute Force) |
| Wednesday | 2017-07-05 | **DoS Hulk, GoldenEye, Slowloris, Slowhttptest, Heartbleed** |
| Thursday | 2017-07-06 | **Web Attacks (XSS, SQL Injection, Brute Force), Infiltration** |
| Friday | 2017-07-07 | **DDoS, PortScan, Bot** |

**Key Observations:**
1. **Wednesday and Friday** have the highest attack volumes (DoS and DDoS are volumetric)
2. **Monday's pure-benign traffic** provides an excellent baseline for anomaly detection
3. **Thursday's attacks** are low-volume but high-impact (Web attacks, Infiltration)
4. The attack schedule means models trained on one day may not generalize to others — **temporal validation splits** should be considered

---
## 6. Correlation Analysis

Identifying **highly correlated features** is crucial for:
- **Dimensionality reduction** — remove redundant features
- **Multicollinearity avoidance** — important for linear models
- **Feature selection** — keep the most informative, drop the rest

In [ ]:
# ── 6a: Select top 20 numeric features & compute correlation ────────
corr_features = [
    'flow_duration', 'total_fwd_packets', 'total_backward_packets',
    'total_length_of_fwd_packets', 'total_length_of_bwd_packets',
    'fwd_packet_length_mean', 'bwd_packet_length_mean',
    'flow_bytes_s', 'flow_packets_s',
    'flow_iat_mean', 'flow_iat_std',
    'fwd_header_length', 'bwd_header_length',
    'packet_length_mean', 'packet_length_std',
    'total_bytes', 'total_packets',
    'bytes_per_packet', 'bytes_per_second', 'packets_per_second'
]

# Sample 100K rows for performance
np.random.seed(42)
corr_sample = df[corr_features].sample(100_000, random_state=42)
print(f'Computing correlation matrix on 100K sample × {len(corr_features)} features')

corr_matrix = corr_sample.corr()

# Mask upper triangle
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='RdYlBu_r',
    center=0,
    linewidths=0.5,
    linecolor='#2a2a4a',
    ax=ax,
    vmin=-1, vmax=1,
    annot_kws={'fontsize': 8},
    cbar_kws={'label': 'Pearson Correlation', 'shrink': 0.8}
)

ax.set_title('Feature Correlation Matrix (20 Key Features, 100K Sample)',
             fontsize=16, fontweight='bold', pad=20)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(fontsize=9)
plt.tight_layout()
fig.savefig(FIG_DIR / '03_correlation_heatmap.png', **SAVE_KW)
plt.show()

In [ ]:
# ── 6b: Identify highly correlated pairs (|r| > 0.9) ───────────────
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.9:
            high_corr_pairs.append((
                corr_matrix.columns[i],
                corr_matrix.columns[j],
                round(r, 3)
            ))

high_corr_df = pd.DataFrame(high_corr_pairs, columns=['Feature A', 'Feature B', 'Correlation'])
high_corr_df = high_corr_df.sort_values('Correlation', ascending=False, key=abs)

print(f'Found {len(high_corr_df)} feature pairs with |r| > 0.9:\n')
print(high_corr_df.to_string(index=False))

### 🔍 Insight — Feature Correlations

**Highly Correlated Pairs (|r| > 0.9):**
- `total_fwd_packets` ↔ `fwd_header_length` — Header length grows linearly with packet count
- `total_length_of_fwd_packets` ↔ `total_bytes` — Forward bytes dominate total bytes
- `fwd_packet_length_mean` ↔ `packet_length_mean` — Forward direction dominates average
- `flow_bytes_s` ↔ `bytes_per_second` — These are essentially the same feature

**Recommendations for ML Phase:**
1. **Drop one feature from each highly-correlated pair** to reduce multicollinearity
2. Consider **PCA** or **feature importance-based selection** for further reduction
3. Our engineered features (`bytes_per_packet`, `packets_per_second`) add signal beyond raw metrics

---
## 7. Feature Distributions by Attack Family

Each attack type has a distinctive **behavioral fingerprint** in feature space.  
Visualizing these differences validates that ML models can learn to distinguish them.

In [ ]:
# ── 7a: Violin plots — Key features by attack family ────────────────
violin_features = ['flow_duration', 'total_bytes', 'packets_per_second', 'bytes_per_packet']

# Use top 5 families + benign sample for clarity
top5_families = ['DoS', 'DDoS', 'PortScan', 'Brute Force', 'Web Attack']
violin_data = attacks[attacks['attack_family'].isin(top5_families)].copy()

# Add a benign sample
benign_sub = df[~df['is_attack']].sample(50_000, random_state=42).copy()
benign_sub['attack_family'] = 'BENIGN'
violin_data = pd.concat([benign_sub, violin_data])

family_palette = {'BENIGN': '#00d2ff', **FAMILY_COLORS}
family_display_order = ['BENIGN'] + top5_families

fig, axes = plt.subplots(2, 2, figsize=(18, 13))

for ax, feat in zip(axes.flat, violin_features):
    # Use log1p for skewed features
    plot_col = f'log_{feat}'
    violin_data[plot_col] = np.log1p(violin_data[feat].clip(lower=0))
    
    sns.violinplot(
        data=violin_data, x='attack_family', y=plot_col,
        order=family_display_order,
        palette=family_palette,
        ax=ax, inner='quartile', linewidth=0.8, cut=0,
        scale='width'
    )
    
    nice_name = feat.replace('_', ' ').title()
    ax.set_title(f'log₁ₚ({nice_name})', fontsize=13, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.tick_params(axis='x', rotation=25)
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('Feature Distributions by Attack Family (Violin Plots, Log-Transformed)',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(FIG_DIR / '03_violin_by_family.png', **SAVE_KW)
plt.show()

In [ ]:
# ── 7b: KDE overlay — Packets per second by family ──────────────────
fig, ax = plt.subplots(figsize=(14, 7))

for fam in top5_families:
    data = attacks[attacks['attack_family'] == fam]['packets_per_second'].clip(upper=attacks['packets_per_second'].quantile(0.99))
    if len(data) > 0:
        data_log = np.log1p(data)
        data_log.plot.kde(ax=ax, label=fam, color=FAMILY_COLORS[fam], linewidth=2)

# Add benign
benign_pps = benign_sub['packets_per_second'].clip(upper=df['packets_per_second'].quantile(0.99))
np.log1p(benign_pps).plot.kde(ax=ax, label='BENIGN', color='#00d2ff', linewidth=2, linestyle='--')

ax.set_title('Packets Per Second Distribution (KDE, Log-Transformed)',
             fontsize=15, fontweight='bold')
ax.set_xlabel('log₁ₚ(Packets Per Second)')
ax.set_ylabel('Density')
ax.legend(fontsize=11, framealpha=0.7)
ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(FIG_DIR / '03_kde_packets_per_second.png', **SAVE_KW)
plt.show()

### 🔍 Insight — Attack Family Fingerprints

Each attack family has a **distinctive profile** in feature space:

| Family | Flow Duration | Bytes Volume | Packets/s | Bytes/Packet |
|--------|--------------|-------------|-----------|-------------|
| **DoS** | Very short (bursts) | High | Very high | Medium |
| **DDoS** | Short–medium | High | Very high | Small |
| **PortScan** | Very short | Very low | Low | Minimal |
| **Brute Force** | Medium–long | Low–medium | Low | Small–medium |
| **Web Attack** | Medium | Medium | Low | Medium–large |
| **BENIGN** | Varied | Varied | Low–medium | Varied |

**This is excellent news for ML models** — the feature distributions are sufficiently different across families that tree-based classifiers (Random Forest, XGBoost) should achieve strong separation.

---
## 8. Outlier Analysis

Outliers in network traffic can indicate either:
- **Attack behavior** (legitimate outliers we want to detect)
- **Data quality issues** (infinite values, sensor errors)

Understanding the outlier landscape helps choose appropriate anomaly detection algorithms.

In [ ]:
# ── 8a: Boxplots for key features ───────────────────────────────────
outlier_features = ['flow_bytes_s', 'flow_duration', 'total_packets']

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, feat in zip(axes, outlier_features):
    # Use log-transformed for better visibility
    data = df[feat].replace([np.inf, -np.inf], np.nan).dropna()
    data_log = np.log1p(data.clip(lower=0))
    
    bp = ax.boxplot(data_log.values, vert=True, patch_artist=True,
                    boxprops=dict(facecolor='#e94560', alpha=0.7),
                    medianprops=dict(color='yellow', linewidth=2),
                    whiskerprops=dict(color='white'),
                    capprops=dict(color='white'),
                    flierprops=dict(marker='o', markerfacecolor='#ff6f61', markersize=2, alpha=0.3))
    
    nice_name = feat.replace('_', ' ').title()
    ax.set_title(f'log₁ₚ({nice_name})', fontsize=13, fontweight='bold')
    ax.set_xticklabels([nice_name], fontsize=10)
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('Outlier Distribution in Key Features (Log-Transformed)',
             fontsize=16, fontweight='bold', y=1.03)
plt.tight_layout()
fig.savefig(FIG_DIR / '03_outlier_boxplots.png', **SAVE_KW)
plt.show()

In [ ]:
# ── 8b: IQR outlier counts per feature ──────────────────────────────
outlier_report = []
for feat in outlier_features:
    data = df[feat].replace([np.inf, -np.inf], np.nan).dropna()
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_outliers = ((data < lower) | (data > upper)).sum()
    pct = n_outliers / len(data) * 100
    outlier_report.append({
        'Feature': feat,
        'Q1': f'{Q1:,.2f}',
        'Q3': f'{Q3:,.2f}',
        'IQR': f'{IQR:,.2f}',
        'Lower Fence': f'{lower:,.2f}',
        'Upper Fence': f'{upper:,.2f}',
        'Outliers': f'{n_outliers:,}',
        'Outlier %': f'{pct:.2f}%'
    })

outlier_df = pd.DataFrame(outlier_report)
print('📊 IQR-Based Outlier Analysis:\n')
print(outlier_df.to_string(index=False))

In [ ]:
# ── 8c: Scatter plot — Total Bytes vs Flow Duration (colored by attack) ──
# Sample 80K for scatter visibility
np.random.seed(42)
scatter_sample = df.sample(80_000, random_state=42).copy()
print(f'Scatter plot using 80K sample ({scatter_sample["is_attack"].sum():,} attacks, '
      f'{(~scatter_sample["is_attack"]).sum():,} benign)')

fig, ax = plt.subplots(figsize=(14, 9))

# Plot benign first (background)
benign_s = scatter_sample[~scatter_sample['is_attack']]
attack_s = scatter_sample[scatter_sample['is_attack']]

ax.scatter(np.log1p(benign_s['flow_duration']), np.log1p(benign_s['total_bytes']),
           c='#00d2ff', alpha=0.15, s=5, label='Benign', rasterized=True)
ax.scatter(np.log1p(attack_s['flow_duration']), np.log1p(attack_s['total_bytes']),
           c='#e94560', alpha=0.4, s=8, label='Attack', rasterized=True, zorder=5)

ax.set_xlabel('log₁ₚ(Flow Duration)', fontsize=12)
ax.set_ylabel('log₁ₚ(Total Bytes)', fontsize=12)
ax.set_title('Total Bytes vs Flow Duration (80K Sample, Log-Transformed)',
             fontsize=15, fontweight='bold')
ax.legend(fontsize=12, markerscale=3, framealpha=0.7)
ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(FIG_DIR / '03_scatter_bytes_vs_duration.png', **SAVE_KW)
plt.show()

### 🔍 Insight — Outlier Analysis

1. **Extreme outliers are pervasive:** A significant percentage of flows exceed the IQR-based upper fence, especially in `flow_bytes_s` and `total_packets`.
2. **Attacks cluster in distinct regions** of the bytes-vs-duration space:
   - **Bottom-left cluster:** PortScan (short duration, few bytes)
   - **Top-left cluster:** DoS/DDoS (short duration, many bytes = high throughput)
   - **Right-spread cluster:** Slowloris/Slowhttptest (long duration, moderate bytes)
3. **Isolation Forest** is well-suited here because:
   - It excels at detecting points in **low-density regions**
   - It handles **multi-modal** outlier distributions
   - It doesn't assume a specific distribution shape

**Business Implication:** Anomaly detection systems should monitor multiple feature dimensions simultaneously — no single threshold captures all attack types.

---
## 9. TCP Flag Analysis

TCP flags are **protocol-level indicators** that reveal connection behavior.  
Different attack types manipulate flags in characteristic ways:
- **SYN floods** → many connection initiations, few completions
- **RST floods** → abnormal connection resets
- **PSH/ACK patterns** → data transfer anomalies

In [ ]:
# ── 9a: Mean flag counts by attack family (stacked bar) ─────────────
flag_cols = ['fin_flag_count', 'rst_flag_count', 'psh_flag_count', 'ack_flag_count',
             'urg_flag_count', 'ece_flag_count']

# Add benign as a "family" for comparison
df_with_family = df.copy()
df_with_family.loc[~df_with_family['is_attack'], 'attack_family'] = 'BENIGN'

flag_means = df_with_family.groupby('attack_family')[flag_cols].mean()
display_families = ['BENIGN'] + list(family_counts.sort_values(ascending=False).index)
flag_means = flag_means.reindex([f for f in display_families if f in flag_means.index])

# Clean column names for display
flag_display = flag_means.rename(columns={
    'fin_flag_count': 'FIN', 'rst_flag_count': 'RST', 'psh_flag_count': 'PSH',
    'ack_flag_count': 'ACK', 'urg_flag_count': 'URG', 'ece_flag_count': 'ECE'
})

flag_colors = ['#e94560', '#ff6f61', '#00d2ff', '#ffc300', '#7b2ff7', '#00e676']

fig, ax = plt.subplots(figsize=(14, 8))
flag_display.plot(kind='bar', stacked=True, ax=ax, color=flag_colors,
                  edgecolor='white', linewidth=0.3)

ax.set_title('Mean TCP Flag Counts by Attack Family', fontsize=15, fontweight='bold')
ax.set_ylabel('Mean Flag Count per Flow')
ax.set_xlabel('Attack Family')
ax.legend(title='TCP Flag', bbox_to_anchor=(1.02, 1), loc='upper left', framealpha=0.7)
ax.tick_params(axis='x', rotation=35)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
fig.savefig(FIG_DIR / '03_flag_analysis_by_family.png', **SAVE_KW)
plt.show()

print('\n📊 Mean Flag Counts by Family:')
print(flag_display.round(3).to_string())

In [ ]:
# ── 9b: Heatmap — Normalized flag patterns ──────────────────────────
# Normalize each row to show relative flag composition
flag_norm = flag_display.div(flag_display.sum(axis=1), axis=0).fillna(0)

fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(
    flag_norm,
    annot=True, fmt='.2f',
    cmap='YlOrRd',
    linewidths=0.5,
    linecolor='#2a2a4a',
    ax=ax,
    vmin=0, vmax=1,
    cbar_kws={'label': 'Proportion of Total Flags'}
)
ax.set_title('TCP Flag Composition by Attack Family (Row-Normalized)',
             fontsize=14, fontweight='bold')
ax.set_ylabel('Attack Family')
ax.set_xlabel('TCP Flag')
plt.tight_layout()
fig.savefig(FIG_DIR / '03_flag_composition_heatmap.png', **SAVE_KW)
plt.show()

### 🔍 Insight — TCP Flag Patterns

1. **PortScan** shows distinctive RST flag patterns — failed connection attempts result in TCP RST responses.
2. **DoS attacks** show high PSH+ACK flags — rapid data pushing to overwhelm the server.
3. **Brute Force** (FTP/SSH-Patator) shows moderate ACK flags — each login attempt completes a TCP handshake.
4. **Benign traffic** has balanced flag distribution reflecting normal connection lifecycle (SYN → ACK → PSH → FIN).
5. **ECE flags** are rare across all families — ECN (Explicit Congestion Notification) is not commonly exploited.

**For ML:** TCP flag features provide strong discriminative signals, especially for PortScan detection. Consider creating a `rst_to_ack_ratio` feature.

---
## 10. Derived Feature Analysis

In the data cleaning phase, we engineered several **derived features** that should provide additional discriminative power:
- `bytes_per_packet` — Average payload size
- `packets_per_second` — Flow rate
- `forward_byte_ratio` — Directional asymmetry

In [ ]:
# ── 10a: Distribution of derived features ───────────────────────────
derived_features = ['bytes_per_packet', 'packets_per_second', 'forward_byte_ratio']

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, feat in zip(axes, derived_features):
    data = df[feat].replace([np.inf, -np.inf], np.nan).dropna()
    
    if feat == 'forward_byte_ratio':
        # This is bounded [0, 1], no log needed
        ax.hist(data.values, bins=50, color='#e94560', edgecolor='white',
                linewidth=0.3, alpha=0.8)
        nice = 'Forward Byte Ratio'
    else:
        # Log transform for skewed distributions
        ax.hist(np.log1p(data.clip(lower=0).values), bins=80, color='#e94560',
                edgecolor='white', linewidth=0.3, alpha=0.8)
        nice = f'log₁ₚ({feat.replace("_", " ").title()})'
    
    ax.set_title(nice, fontsize=13, fontweight='bold')
    ax.set_ylabel('Count')
    ax.grid(axis='y', alpha=0.3)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}K' if x >= 1000 else f'{x:.0f}'))

fig.suptitle('Derived Feature Distributions (Full Dataset)',
             fontsize=16, fontweight='bold', y=1.03)
plt.tight_layout()
fig.savefig(FIG_DIR / '03_derived_feature_distributions.png', **SAVE_KW)
plt.show()

In [ ]:
# ── 10b: Derived features by top 5 attack families ──────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 7))

compare_families = ['BENIGN'] + top5_families

for ax, feat in zip(axes, derived_features):
    data_list = []
    labels_list = []
    colors_list = []
    
    for fam in compare_families:
        if fam == 'BENIGN':
            vals = benign_sub[feat].replace([np.inf, -np.inf], np.nan).dropna()
        else:
            mask = attacks['attack_family'] == fam
            vals = attacks.loc[mask, feat].replace([np.inf, -np.inf], np.nan).dropna()
        
        if len(vals) > 10000:
            vals = vals.sample(10000, random_state=42)
        
        data_list.append(vals.values)
        labels_list.append(fam)
        colors_list.append(family_palette.get(fam, '#888'))
    
    bp = ax.boxplot(data_list, vert=True, patch_artist=True, labels=labels_list,
                    flierprops=dict(marker='.', markersize=1, alpha=0.2))
    for patch, color in zip(bp['boxes'], colors_list):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    for median in bp['medians']:
        median.set_color('yellow')
        median.set_linewidth(2)
    
    nice_name = feat.replace('_', ' ').title()
    ax.set_title(nice_name, fontsize=13, fontweight='bold')
    ax.tick_params(axis='x', rotation=30)
    ax.grid(axis='y', alpha=0.3)
    
    # Use log scale for heavily skewed features
    if feat != 'forward_byte_ratio':
        ax.set_yscale('symlog', linthresh=1)

fig.suptitle('Derived Features by Attack Family (Top 5 + Benign)',
             fontsize=16, fontweight='bold', y=1.03)
plt.tight_layout()
fig.savefig(FIG_DIR / '03_derived_features_by_family.png', **SAVE_KW)
plt.show()

### 🔍 Insight — Derived Feature Value

1. **`bytes_per_packet`:** 
   - PortScan has near-zero bytes per packet (empty SYN probes)
   - Web attacks show higher values (carrying HTTP payloads)
   - Strong separator between scan-based and payload-based attacks

2. **`packets_per_second`:**
   - DoS/DDoS show extremely high packet rates (>10K packets/s)
   - Brute Force has moderate rates (systematic login attempts)
   - Excellent feature for volumetric attack detection

3. **`forward_byte_ratio`:**
   - Benign traffic shows balanced bidirectional communication (~0.5)
   - DoS attacks show asymmetric ratios (attacker sends most data)
   - PortScan shows high forward ratio (requests without responses)

**Conclusion:** These engineered features provide **strong discriminative power** that raw features alone may not capture. They should be included in the ML feature set.

---
## 11. Key Findings Summary

### Top 10 Analytical Insights from EDA

1. **Extreme class imbalance:** BENIGN accounts for 83.5% of traffic. The top 3 attack families (DoS, DDoS, PortScan) make up 96.8% of all attacks, while Infiltration (36) and Heartbleed (11) are ultra-rare.

2. **DoS dominates the threat landscape:** With 193K+ flows (~45% of attacks), Denial-of-Service is the most prevalent attack category. This mirrors real-world patterns where volumetric attacks are most common.

3. **Attack traffic has distinctive behavioral signatures:** Key differentiators include `bytes_per_second` (very high for DoS/DDoS), `flow_duration` (bimodal — very short or very long), and `bytes_per_packet` (near-zero for PortScan).

4. **Port 80 (HTTP) is the primary target:** Over 324K attack flows target port 80, receiving every attack type. Ports 21 (FTP) and 22 (SSH) are targeted specifically by brute force attacks.

5. **Temporal attack campaign:** Different attack types were executed on different days, with Monday as the benign baseline. This creates a natural temporal validation opportunity but also means attack patterns are date-dependent.

6. **High feature correlation exists:** Multiple feature pairs have |r| > 0.9 (e.g., `total_fwd_packets` ↔ `fwd_header_length`). Feature selection is needed before training linear models.

7. **Each attack family has a unique fingerprint:** Violin plots and KDE analysis confirm that feature distributions vary significantly across families, validating that multi-class classification is feasible.

8. **Outliers are meaningful, not noise:** Extreme values in `flow_bytes_s` and `total_packets` correspond to actual attack behavior. Isolation Forest anomaly detection is well-suited for this data.

9. **TCP flag patterns are discriminative:** PortScan shows high RST flags, DoS shows high PSH+ACK patterns. Flag features should be retained for ML.

10. **Engineered features add value:** `bytes_per_packet`, `packets_per_second`, and `forward_byte_ratio` show strong separation across attack families, justifying the feature engineering effort.

---

### Recommendations for ML Phase

| Decision | Recommendation | Rationale |
|----------|---------------|----------|
| **Class Imbalance** | Use SMOTE or class weights | Rare attacks (Heartbleed, Infiltration) need upsampling |
| **Feature Selection** | Drop one from each correlated pair | Reduce from ~90 to ~30 features |
| **Model Choice** | Start with Random Forest + XGBoost | Tree models handle imbalanced, mixed-scale data well |
| **Anomaly Detection** | Isolation Forest on engineered features | Effective for the multi-modal outlier structure |
| **Validation** | Stratified K-Fold, consider temporal splits | Preserve rare class proportions |
| **Evaluation Metric** | F1-Macro (not accuracy) | Accuracy is misleading with 83.5% majority class |

In [ ]:
# ── Final summary stats ─────────────────────────────────────────────
print('=' * 65)
print('  EXPLORATORY DATA ANALYSIS — COMPLETE')
print('=' * 65)
print(f'  Dataset:         {df.shape[0]:>12,} rows × {df.shape[1]} columns')
print(f'  Benign flows:    {(~df["is_attack"]).sum():>12,} ({(~df["is_attack"]).mean():.1%})')
print(f'  Attack flows:    {df["is_attack"].sum():>12,} ({df["is_attack"].mean():.1%})')
print(f'  Attack labels:   {df.loc[df["is_attack"], "observed_label"].nunique():>12}')
print(f'  Attack families: {df["attack_family"].nunique():>12}')
print(f'  Capture dates:   {df["capture_date"].nunique():>12}')
print(f'  Figures saved:   ../reports/figures/03_*.png')
print('=' * 65)
print('\n📌 Next: Notebook 04 — Feature Engineering & Selection')

# Clean up large objects
del sample_df, benign_sample, attack_sample, scatter_sample, violin_data
del benign_sub, corr_sample, df_with_family
import gc; gc.collect()
print('\n🧹 Memory cleaned up.')